# 📌 Lottery Ticket Hypothesis

![Topic](https://img.shields.io/badge/Topic-Lottery Ticket Hypothesis-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-Architecture-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-April%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — A large neural network contains small, sparse sub-networks ("winning tickets") that can match the full network's accuracy, when trained in isolation from the start. Meaning that most of the network's weights are dead weight.</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install torch torch.nn copy` |


---
## 1. Overview
In 2019, Jonathan Frankle and Michael Carlin published ‘The Lottery Ticket Hypothesis’. In their analysis, they argue that when we train a massive neural network, we are in fact taking part in a lottery. 
<br>
Their theory explains that there is a tiny sub-network with a lucky initialisation hidden at the heart of this vast network. If this single sub-network had been trained from scratch with exactly the same starting weights, it would have achieved results just as good as the overall network, impling that the rest of the network is useless noise.

This paper therefore challenges the assumption that you need massive networks to train effectively


---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

The core procedure is called Iterative Magnitude Pruning (IMP), it :

1. Initialize a large network with random weights (save these as θ₀)
2. Train it fully until convergence
3. Prune the p% of weights with the smallest absolute value (set them to zero)
4. Rewind the remaining weights back to their original values at θ₀ (not random — the same original values)
5. Retrain this sparse network from scratch
6. Repeat until the network is small enough

![Lottery_Ticket_Hypothesis](../assets/Lottery_Ticket_Hypothesis.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | Extreme compression | Enables massive model compression (up to 90%+ of weights removed) |
| 🟢 | Edge deployment | Opens doors to efficient inference on edge devices |
| 🟢 | Overparameterization insight | Challenges the belief that big networks are necessary |
| 🟢 | Research catalyst | nspires new directions in neural architecture search and transfer learning |
| 🔴 | Search cost | Finding the ticket is expensive because you still must train the full network first |
| 🔴 | Scale fragility | Doesn't scale easily to very large models (like LLMs) without approximations |
| 🔴 | Ticket specificity | Winning tickets are often dataset- and task-specific; they don't transfer universally |
| 🔴 | Rewind sensitivity | The "rewind" step is crucial and sensitive — small changes break the ticket |

---
## 4. Code Example

> **Goal:** Build an example of Lottery Ticket Hypothesis

In [ ]:
# ---------------------------------------------------------------------------
# BUILD A SIMPLE NETWORK & SAVE ORIGINAL WEIGHTS
# ---------------------------------------------------------------------------
import torch
import torch.nn as nn
import copy

class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 300)
        self.fc2 = nn.Linear(300, 100)
        self.fc3 = nn.Linear(100, 10)

    def forward(self, x):
        x = nn.functional.relu(self.fc1(x))
        x = nn.functional.relu(self.fc2(x))
        return self.fc3(x)

model = SimpleNet()
original_weights = copy.deepcopy(model.state_dict())

# ---------------------------------------------------------------------------
# TRAIN THE FULL NETWORK
# ---------------------------------------------------------------------------
train(model, train_loader, epochs=5)
print(f"Full model accuracy: {evaluate(model):.2%}")

# ---------------------------------------------------------------------------
# PRUNE: ZERO OUT THE SMALLEST P% OF WEIGHTS
# ---------------------------------------------------------------------------
def prune_by_magnitude(model, prune_ratio=0.2):
    """Set the lowest-magnitude weights to zero (create a mask)."""
    masks = {}
    for name, param in model.named_parameters():
        if 'weight' in name:
            threshold = torch.quantile(param.data.abs(), prune_ratio)
            # Mask = 1 where weight survives, 0 where pruned
            masks[name] = (param.data.abs() >= threshold).float()
            param.data *= masks[name]  # apply mask in-place
    return masks

masks = prune_by_magnitude(model, prune_ratio=0.5)

# ---------------------------------------------------------------------------
# REWIND: RESTORE ORIGINAL WEIGHTS, KEEP MASK
# ---------------------------------------------------------------------------
def rewind_to_init(model, original_weights, masks):
    """Reset surviving weights to their original (lucky) values."""
    model.load_state_dict(original_weights)
    for name, param in model.named_parameters():
        if name in masks:
            param.data *= masks[name]

rewind_to_init(model, original_weights, masks)

# ---------------------------------------------------------------------------
# RETRAIN THE SPARSE "WINNING TICKET"
# ---------------------------------------------------------------------------
train(model, train_loader, epochs=5, masks=masks)
print(f"Winning ticket accuracy: {evaluate(model):.2%}")

# FIX: division was split across two lines — now on one line
sparsity = 1 - sum(m.sum() for m in masks.values()) / sum(m.numel() for m in masks.values())
print(f"Network sparsity: {sparsity:.1%}")

---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **Large networks** are a **lottery**, not a guarantee. Most weights are dead weight from the start.
- **Initialization** is more powerful than architecture
- **Rewinding** is what **separates** a **winning ticket** from **ordinary pruning**.
- **Compression is the reward**, but the search cost is real.
- **Winning tickets** are task-specific, not universal

</div>